In [2]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
docs = (
    PyPDFLoader("../documents/Company_Policies.pdf").load() + 
    PyPDFLoader("../documents/Company_Profile.pdf").load() + 
    PyPDFLoader("../documents/Product_and_Pricing.pdf").load() 
)

In [6]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, 
    chunk_overlap=150, 
).split_documents(docs)

In [7]:
len(chunks)

16

In [9]:
embeddings = OllamaEmbeddings(model='bge-m3')
vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings, 
    persist_directory="../vector-database/selfrag1"
)

In [10]:
retriever = vector_store.as_retriever(search_kwargs={'k':4})

In [11]:
llm = ChatOllama(model="llama3.2", temperature=0)

In [12]:
#Graph state
class State(TypedDict): 
        question: str 
        need_retrieval: bool
        docs: List[Document]

        answer: str

In [13]:
class RetrieveDecision(BaseModel): 
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed ti answer reliably, else False,"
    )

In [15]:
decide_retrieve_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You decide whether document retrieval is necessary.

            Return JSON matching this schema:
            {{
                "should_retrieve": boolean
            }}

            Guidelines:
            - should_retrieve = true if answering requires
              specific facts, citations, or information
              that may not be available in the model's knowledge.

            - should_retrieve = false for general explanations,
              definitions, or reasoning that does not require
              external sources.

            - If unsure, choose true.
            """
        ),
        (
            "human",
            "Question: {question}"
        )
    ]
)
# Important: no '.content' for structured output
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)



In [16]:
def decide_retrieval(state: State): 
    decision: RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieve_prompt.format_messages(question=state['question'])
    )
    return {"need_retrieval":decision.should_retrieve}

In [ ]:

direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            Answer the question using only your general knowledge.

            Do NOT assume access to external documents.

            If you are unsure or the answer requires specific
            sources, say:

            I don't know based on my general knowledge.
            """
        ),
        ("human", "{question}")
    ]
)


def generate_direct(state: State):

    out = llm.invoke(
        direct_generation_prompt.format_messages(
            question=state["question"]
        )
    )

    return {
        "answer": out.content
    }